In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [3]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [4]:
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
response.choices[0].message.content

'Hi there! Welcome, and nice to meet you. I’m here to help with questions, writing, coding, studying, brainstorming, or just chat. What would you like to do today? Tell me a bit about your interests or a task you want help with, and I’ll tailor my responses.'

In [5]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Home
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
We work with groundbreaking, proprietary LLMs verticalized for talent, we’ve
patented
our matching model, and our award-winning platform has happy customers and tons of press coverage.
Conne

## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [6]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [7]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
response.choices[0].message.content

'2 + 2 equals 4.'

In [9]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [10]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nHome\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n.

In [11]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [12]:
summarize("https://edwarddonner.com")

'# Edward Donner’s Playground for AI Nerds and DJ Wannabes 🎧🤖\n\nMeet Ed, a code-slinging, LLM-tinkering CTO who moonlights as an amateur electronic music producer and professional nodder at Hacker News threads he barely understands. He’s the brains behind Nebula.io, where AI magically matches recruiters with talent, armed with patented tech and shiny awards.\n\nOn this site, you can watch large language models duke it out in “Outsmart,” an AI arena for sax-sounding diplomacy battles, or waste some time playing Connect Four. If you want Ed’s hot takes, check his posts on AI live events, AWS-powered agentic AI, and how to be an AI engineer-slash-leader.\n\nLooking to slide into his inbox? Just decode his email and hit him up. Prefer stalking over emailing? He’s on LinkedIn, Twitter, and Facebook. Want mind-blowing AI insights delivered to your inbox? Subscribe to his newsletter and prepare to be unbored.'

In [13]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [14]:
display_summary("https://edwarddonner.com")

# Edward Donner’s Nerd Den

So, here we have Ed—a code monkey who moonlights as a (not-so-hip) DJ and a reluctant devotee of Hacker News. He’s the brains behind some fancy AI startup Nebula.io that’s basically matchmaking for recruiters and job seekers, powered by proprietary LLM magic and patented algorithms. 

Oh, and Ed used to run another AI startup called untapt, which got snatched up in 2021. Talk about an exit!

News? Yup, Ed’s dropping some knowledge bombs:
- Nov 2025: The mystical vibes of AI live events
- Sept 2025: AI in real-world muscle—Gen AI and Agentic AI on AWS
- May 2025 (two posts): Stay AI-savvy with his engineer-leader curriculum and executive briefing

He also has a playground called "Outsmart" where LLMs battle it out with diplomacy and deviousness—basically AI soap opera but geekier.

Want to geek out or pitch him? Ed’s just an email away, or follow his LinkedIn/Twitter/Facebook fan club. Newsletter sign-up included, because who doesn’t need another AI update spam?

In [15]:
display_summary("https://cnn.com")

# CNN: The Place Where News Never Takes a Coffee Break

Welcome to CNN, your one-stop shop for all the chaos and curiosities happening around the world — from US drama to international sagas, business blips to celebrity slip-ups. They’ve got everything: politics, sports, the climate meltdown, and even the latest in pickleball (okay, maybe not pickleball, but close).

And if you're annoyed by ads that slow your scroll or freeze mid-fizzle, CNN *totally* wants your feedback because apparently, your pain is their gain.

No juicy headlines or big announcements here—just the usual news buffet served 24/7, with enough categories (and ads) to keep you endlessly distracted. So grab your popcorn and enjoy the endless stream of "Breaking News" you didn't know you needed!

In [16]:
display_summary("https://anthropic.com")

# Anthropic Website Summary: AI with a Conscience

Anthropic is proudly playing the role of AI’s good guy, pushing research and products that prioritize safety and long-term human well-being. They’re all about “securing benefits and mitigating risks,” which basically means trying to make AI less Skynet and more helpful sidekick.

The star of the show? Claude — their AI lineup includes the latest and greatest models like Claude Sonnet 4.5 (apparently the “best model in the world for agents, coding, and computer use”) and Claude Haiku 4.5. If you’re curious about the tech or want to tinker, they’ve got a developer platform, docs, and even an academy (because who doesn’t want to learn AI with a side of responsible scaling policy?).

They also emphasize transparency, trust, security, and ethics — so they’re not just tossing out AI models and hoping for the best. News and announcements focus on the rollout of these new Claude versions, hyped as big deals for coding and agent tasks.

In short: AI heroes with a heart, juggling cutting-edge models and careful introspection, all while trying to keep the robot apocalypse on hold.

In [17]:
# Step 1: Create your prompts

system_prompt = "something here"
user_prompt = """
    Lots of text
    Can be pasted here
"""

# Step 2: Make the messages list

messages = [] # fill this in

# Step 3: Call OpenAI
# response =

# Step 4: print the result
# print(